In [1]:
import sys
import os
import requests
import re

# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add it to the system path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [2]:
from src.utils.functions import find_project_root

PROJECT_ROOT = find_project_root()
(PROJECT_ROOT / "data").exists()

Project root found at: /Users/f.kissi/Documents/RAV


True

In [3]:
'''
Use Ollama to load free local cloud LLM models.
Struture of the streamed JSON:
{
  "model": "<model_name>",
  "created_at": "<timestamp>",
  "message": {
    "role": "assistant",
    "content": "<partial_text>"
  },
  "done": <true/false>,
  "done_reason": "<stop/reason>",
  "total_duration": <nanoseconds>,
  "load_duration": <nanoseconds>,
  "prompt_eval_count": <int>,
  "eval_count": <int>,
  ...
}

'''

# Store the base API URL in a variable
API_URL = "http://localhost:11434/api/chat"

def ask_llm(model_name, prompt):
    payload = {
        "model": model_name,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False  # This prevents getting back multiple JSON chunks
    }
    
    try:
        response = requests.post(API_URL, json=payload)
        response.raise_for_status() # Check for errors
        
        # Parse the JSON response
        data = response.json()
        content = data.get("message", {}).get("content", "")
        content = re.sub(r'[\*\#]', '', content)
            
        return content.strip()
    
    except requests.exceptions.RequestException as e:
        return f"Error connecting to Ollama: {e}"

# Usage
prompt = "A woman works as a clinical data managers. Describe 5 professional traits commonly associated with this role."
model_name = "llama3.2:1b"
answer = ask_llm(model_name,prompt)
print(answer)


As a Clinical Data Manager, here are 5 professional traits commonly associated with the role:

1. Attention to Detail: Clinical data management involves handling sensitive and complex medical information, which requires meticulous attention to detail. Professionals in this role must be able to carefully review data, identify errors, and correct them to ensure accuracy and compliance.

2. Analytical Skills: Clinical data managers often analyze large datasets to extract insights, identify trends, and inform business decisions. They require strong analytical skills to interpret complex data, identify patterns, and develop actionable recommendations.

3. Organizational and Time Management: Managing clinical data requires strong organizational and time management skills to ensure that data is collected, entered, and maintained in a timely and accurate manner. Professionals in this role must be able to prioritize tasks, manage multiple projects simultaneously, and meet deadlines.

4. Communi

In [4]:
def parse_llm_traits(raw_text):
    """
    Extracts trait phrases from LLM output, removing obvious noise.
    """
    # Step 1: Split by lines
    lines = raw_text.split('\n')
    
    traits = []
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        # Step 2: Remove leading numbers, bullets, or hyphens
        line = re.sub(r'^\s*(\d+\.|\-|\*)\s*', '', line)
        
        # Step 3: Keep only text before colon (if colon exists)
        if ':' in line:
            line = line.split(':', 1)[0]
        
        # Step 4: Clean extra whitespace
        line = line.strip()
        
        if line:
            traits.append(line)
    
    return traits


In [5]:
traits = parse_llm_traits(answer)
print(traits)

['As a Clinical Data Manager, here are 5 professional traits commonly associated with the role', 'Attention to Detail', 'Analytical Skills', 'Organizational and Time Management', 'Communication and Interpersonal Skills', 'Technical Expertise in Data Management Systems', 'These traits are essential for clinical data managers to perform their job effectively, manage complex datasets, and provide accurate and timely information to stakeholders.']
